# Spark Partitioning, Shuffle & Broadcast Joins
## Databricks Serverless — Complete PySpark Performance Guide

### Core question
How do Spark partitioning, shuffle, broadcast joins, data skew, and Adaptive Query
Execution affect distributed query performance, and how can we identify and optimize
these bottlenecks?

### Optimization workflow
**Inspect → Diagnose → Optimize → Measure → Validate**

### Serverless compatibility
This notebook is deliberately written for **Databricks Serverless Compute**.

It avoids:
- Python RDD APIs such as `df.rdd.getNumPartitions()`
- manual AQE configuration such as `spark.sql.adaptive.enabled`
- cluster-specific APIs
- unnecessary cache/persist operations

Instead, partitioning and shuffle behavior are inspected through:
- DataFrame APIs
- `explain("formatted")`
- `spark_partition_id()`
- query execution plans
- runtime benchmarks
- result validation

## Project architecture

```text
Synthetic E-commerce Data
          │
          ▼
     Baseline Query
          │
          ▼
   Physical Plan Analysis
          │
     ┌────┼──────────────┐
     ▼    ▼              ▼
Partitioning  Shuffle   Broadcast
     │    │              │
     └────┼──────────────┘
          ▼
       Data Skew
          │
          ▼
      AQE / Salting
          │
          ▼
  Performance Comparison
          │
          ▼
   Practical Guidelines
```

### Concepts demonstrated

| Concept | Demonstration |
|---|---|
| Spark partitions | `spark_partition_id()` |
| `repartition()` | Change partition distribution |
| `coalesce()` | Reduce partitions |
| Shuffle | Identify `Exchange` operators |
| Physical plans | `explain("formatted")` |
| Broadcast joins | Small dimension → large fact |
| Sort-merge joins | Large ↔ large |
| Data skew | Deliberately skew customer keys |
| Salting | Mitigate hot-key skew |
| AQE | Inspect runtime-adaptive physical plans |
| Benchmarking | Compare observed runtimes |
| Correctness | Assertions and reconciliation |

# 1. Environment and imports

### Important Serverless changes

The original notebook attempted to read:

```python
spark.conf.get("spark.sql.adaptive.enabled")
```

That configuration is not available in the current Serverless environment.

It also used:

```python
df.rdd.getNumPartitions()
```

Python RDD APIs are not supported on Serverless.

Therefore this notebook does **not** use either approach.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import broadcast
import time

print("Spark version:", spark.version)
print("Default shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("Environment validation: PASS")

Spark version: 4.1.0
Default shuffle partitions: auto
Environment validation: PASS


# 2. Dataset configuration

We create a controlled synthetic e-commerce workload:

- 10K customers
- 2K products
- 500K orders
- 1.5M order items

The dataset is deterministic so that experiments are reproducible.

In [0]:
NUM_CUSTOMERS = 10_000
NUM_PRODUCTS = 2_000
NUM_ORDERS = 500_000
NUM_ORDER_ITEMS = 1_500_000

SEED = 42

print("Customers :", f"{NUM_CUSTOMERS:,}")
print("Products  :", f"{NUM_PRODUCTS:,}")
print("Orders    :", f"{NUM_ORDERS:,}")
print("Items     :", f"{NUM_ORDER_ITEMS:,}")

Customers : 10,000
Products  : 2,000
Orders    : 500,000
Items     : 1,500,000


## 2.1 Customers

In [0]:
customers = (
    spark.range(1, NUM_CUSTOMERS + 1)
    .withColumnRenamed("id", "customer_id")
    .withColumn(
        "customer_name",
        F.concat(F.lit("Customer_"), F.col("customer_id"))
    )
    .withColumn(
        "city",
        F.element_at(
            F.array(
                F.lit("Chennai"),
                F.lit("Bangalore"),
                F.lit("Hyderabad"),
                F.lit("Mumbai"),
                F.lit("Delhi")
            ),
            ((F.col("customer_id") % 5) + 1).cast("int")
        )
    )
)

display(customers.limit(10))

customer_id,customer_name,city
1,Customer_1,Bangalore
2,Customer_2,Hyderabad
3,Customer_3,Mumbai
4,Customer_4,Delhi
5,Customer_5,Chennai
6,Customer_6,Bangalore
7,Customer_7,Hyderabad
8,Customer_8,Mumbai
9,Customer_9,Delhi
10,Customer_10,Chennai


## 2.2 Products

In [0]:
products = (
    spark.range(1, NUM_PRODUCTS + 1)
    .withColumnRenamed("id", "product_id")
    .withColumn(
        "product_name",
        F.concat(F.lit("Product_"), F.col("product_id"))
    )
    .withColumn(
        "category",
        F.element_at(
            F.array(
                F.lit("Electronics"),
                F.lit("Home"),
                F.lit("Fashion"),
                F.lit("Sports"),
                F.lit("Books")
            ),
            ((F.col("product_id") % 5) + 1).cast("int")
        )
    )
    .withColumn(
        "unit_price",
        F.round(
            F.lit(100.0) + (F.col("product_id") % 500) * 10.0,
            2
        )
    )
)

display(products.limit(10))

product_id,product_name,category,unit_price
1,Product_1,Home,110.0
2,Product_2,Fashion,120.0
3,Product_3,Sports,130.0
4,Product_4,Books,140.0
5,Product_5,Electronics,150.0
6,Product_6,Home,160.0
7,Product_7,Fashion,170.0
8,Product_8,Sports,180.0
9,Product_9,Books,190.0
10,Product_10,Electronics,200.0


## 2.3 Orders

In [0]:
orders = (
    spark.range(1, NUM_ORDERS + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "customer_id",
        ((F.col("order_id") * 17) % NUM_CUSTOMERS) + 1
    )
    .withColumn(
        "order_date",
        F.date_add(
            F.to_date(F.lit("2025-01-01")),
            (F.col("order_id") % 365).cast("int")
        )
    )
    .withColumn(
        "order_status",
        F.element_at(
            F.array(
                F.lit("COMPLETED"),
                F.lit("COMPLETED"),
                F.lit("COMPLETED"),
                F.lit("CANCELLED")
            ),
            ((F.col("order_id") % 4) + 1).cast("int")
        )
    )
)

display(orders.limit(10))

order_id,customer_id,order_date,order_status
1,18,2025-01-02,COMPLETED
2,35,2025-01-03,COMPLETED
3,52,2025-01-04,CANCELLED
4,69,2025-01-05,COMPLETED
5,86,2025-01-06,COMPLETED
6,103,2025-01-07,COMPLETED
7,120,2025-01-08,CANCELLED
8,137,2025-01-09,COMPLETED
9,154,2025-01-10,COMPLETED
10,171,2025-01-11,COMPLETED


## 2.4 Order items

In [0]:
order_items = (
    orders
    .select("order_id")
    .crossJoin(
        spark.range(1, 4)
        .withColumnRenamed("id", "item_number")
    )
    .withColumn(
        "product_id",
        ((F.col("order_id") * 31 + F.col("item_number") * 7)
         % NUM_PRODUCTS) + 1
    )
    .withColumn(
        "quantity",
        ((F.col("order_id") + F.col("item_number")) % 4) + 1
    )
    .select(
        "order_id",
        "item_number",
        "product_id",
        "quantity"
    )
)

display(order_items.limit(10))

assert order_items.count() == NUM_ORDER_ITEMS
print("Order-item generation: PASS")

order_id,item_number,product_id,quantity
1,1,39,3
2,2,77,1
3,3,115,3
4,1,132,2
5,2,170,4
6,3,208,2
7,1,225,1
8,2,263,3
9,3,301,1
10,1,318,4


Order-item generation: PASS


## 2.5 Dataset validation

In [0]:
dataset_counts = {
    "customers": customers.count(),
    "products": products.count(),
    "orders": orders.count(),
    "order_items": order_items.count()
}

for name, count in dataset_counts.items():
    print(f"{name:12s}: {count:,}")

assert dataset_counts["customers"] == NUM_CUSTOMERS
assert dataset_counts["products"] == NUM_PRODUCTS
assert dataset_counts["orders"] == NUM_ORDERS
assert dataset_counts["order_items"] == NUM_ORDER_ITEMS

print("Dataset quality gate: PASS")

customers   : 10,000
products    : 2,000
orders      : 500,000
order_items : 1,500,000
Dataset quality gate: PASS


# 3. Baseline workload

Business question:

> What is the total revenue generated by each customer?

The baseline intentionally avoids:
- explicit `repartition()`
- explicit `coalesce()`
- explicit `broadcast()`
- salting
- manual AQE configuration

We first inspect the natural physical plan.

In [0]:
item_revenue = (
    order_items
    .join(
        products.select("product_id", "unit_price"),
        on="product_id",
        how="inner"
    )
    .withColumn(
        "revenue",
        F.col("quantity") * F.col("unit_price")
    )
)

baseline_customer_revenue = (
    orders
    .join(
        item_revenue,
        on="order_id",
        how="inner"
    )
    .join(
        customers.select(
            "customer_id",
            "customer_name",
            "city"
        ),
        on="customer_id",
        how="inner"
    )
    .groupBy(
        "customer_id",
        "customer_name",
        "city"
    )
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.countDistinct("order_id").alias("order_count")
    )
    .orderBy(F.col("total_revenue").desc())
)

print("=== BASELINE PHYSICAL PLAN ===")
baseline_customer_revenue.explain("formatted")

=== BASELINE PHYSICAL PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (44)
+- == Initial Plan ==
   PhotonResultStage (43)
   +- PhotonColumnarToRow (42)
      +- PhotonSort (41)
         +- PhotonShuffleExchangeSource (40)
            +- PhotonShuffleMapStage (39)
               +- PhotonShuffleExchangeSink (38)
                  +- PhotonGroupingAgg (37)
                     +- PhotonShuffleExchangeSource (36)
                        +- PhotonShuffleMapStage (35)
                           +- PhotonShuffleExchangeSink (34)
                              +- PhotonGroupingAgg (33)
                                 +- PhotonGroupingAgg (32)
                                    +- PhotonGroupingAgg (31)
                                       +- PhotonProject (30)
                                          +- PhotonBroadcastHashJoin Inner (29)
                                             :- PhotonProject (23)
                                             :  +- PhotonBroadcastHashJoin Inner (22)

In [0]:
baseline_result = baseline_customer_revenue.collect()

print("Customers returned:", len(baseline_result))
assert len(baseline_result) == NUM_CUSTOMERS

display(baseline_customer_revenue.limit(20))
print("Baseline query validation: PASS")

Customers returned: 10000


customer_id,customer_name,city,total_revenue,order_count
3133,Customer_3133,Mumbai,2261500.0,50
9133,Customer_9133,Mumbai,2261500.0,50
7133,Customer_7133,Mumbai,2261500.0,50
4133,Customer_4133,Mumbai,2261500.0,50
5633,Customer_5633,Mumbai,2261500.0,50
1633,Customer_1633,Mumbai,2261500.0,50
2633,Customer_2633,Mumbai,2261500.0,50
133,Customer_133,Mumbai,2261500.0,50
3633,Customer_3633,Mumbai,2261500.0,50
5133,Customer_5133,Mumbai,2261500.0,50


Baseline query validation: PASS


## 3.1 Benchmark helper

In [0]:
def benchmark_query(df, label, runs=2):
    timings = []

    # Warm-up
    df.count()

    for run in range(1, runs + 1):
        start = time.perf_counter()
        result_count = df.count()
        elapsed = time.perf_counter() - start

        timings.append(elapsed)

        print(
            f"{label} | Run {run} | "
            f"Rows: {result_count:,} | "
            f"Time: {elapsed:.2f}s"
        )

    return {
        "experiment": label,
        "rows": result_count,
        "min_seconds": min(timings),
        "avg_seconds": sum(timings) / len(timings)
    }

baseline_metrics = benchmark_query(
    baseline_customer_revenue,
    "Baseline Customer Revenue"
)

Baseline Customer Revenue | Run 1 | Rows: 10,000 | Time: 0.65s
Baseline Customer Revenue | Run 2 | Rows: 10,000 | Time: 0.76s


# 4. Serverless-compatible partition inspection

Instead of the unsupported:

```python
df.rdd.getNumPartitions()
```

we inspect partition distribution with the DataFrame function:

```python
spark_partition_id()
```

This lets us observe how rows are distributed without using the RDD API.

In [0]:
def partition_distribution(df, label):
    result = (
        df
        .withColumn("partition_id", F.spark_partition_id())
        .groupBy("partition_id")
        .count()
        .orderBy("partition_id")
    )

    print(f"=== {label} PARTITION DISTRIBUTION ===")
    display(result)

    stats = (
        result
        .agg(
            F.count("*").alias("partition_count"),
            F.min("count").alias("min_rows"),
            F.max("count").alias("max_rows"),
            F.avg("count").alias("avg_rows")
        )
        .first()
    )

    print("Partitions :", stats["partition_count"])
    print("Min rows   :", stats["min_rows"])
    print("Max rows   :", stats["max_rows"])
    print("Avg rows   :", round(stats["avg_rows"], 2))

    return result

orders_distribution = partition_distribution(
    orders,
    "Orders"
)

=== Orders PARTITION DISTRIBUTION ===


partition_id,count
0,62500
1,62500
2,62500
3,62500
4,62500
5,62500
6,62500
7,62500


Partitions : 8
Min rows   : 62500
Max rows   : 62500
Avg rows   : 62500.0


# 5. `repartition()`

`repartition()` changes partition count/distribution and generally introduces a shuffle.

The important thing is not just the number of partitions; inspect the physical plan
to identify the `Exchange` operator.

In [0]:
orders_repartitioned = orders.repartition(8)

print("=== REPARTITION(8) PLAN ===")
orders_repartitioned.explain("formatted")

partition_distribution(
    orders_repartitioned,
    "Orders after repartition(8)"
)

=== REPARTITION(8) PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- == Initial Plan ==
   PhotonResultStage (8)
   +- PhotonColumnarToRow (7)
      +- PhotonShuffleExchangeSource (6)
         +- PhotonShuffleMapStage (5)
            +- PhotonShuffleExchangeSink (4)
               +- PhotonSort (3)
                  +- PhotonProject (2)
                     +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [id#12073L AS order_id#12074L, (((id#12073L * 17) % 10000) + 1) AS customer_id#12076L, date_add(2025-01-01, cast((id#12073L % 365) as int)) AS order_date#12078, element_at([COMPLETED,COMPLETED,COMPLETED,CANCELLED], cast(((id#12073L % 4) + 1) as int), None, true) AS order_status#12080]

(3) PhotonSort
Input [4]: [order_id#12074L, customer_id#12076L, order_date#12078, order_status#12080]
Arguments: [order_id#12074L ASC NULLS FIRST, customer_id#12076L ASC NULLS FIRST, order

partition_id,count
0,62501
1,62500
2,62501
3,62500
4,62499
5,62500
6,62499
7,62500


Partitions : 8
Min rows   : 62499
Max rows   : 62501
Avg rows   : 62500.0


DataFrame[partition_id: int, count: bigint]

In [0]:
orders_by_customer = orders.repartition(8, "customer_id")

print("=== REPARTITION BY CUSTOMER PLAN ===")
orders_by_customer.explain("formatted")

partition_distribution(
    orders_by_customer,
    "Orders repartitioned by customer_id"
)

=== REPARTITION BY CUSTOMER PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (8)
+- == Initial Plan ==
   PhotonResultStage (7)
   +- PhotonColumnarToRow (6)
      +- PhotonShuffleExchangeSource (5)
         +- PhotonShuffleMapStage (4)
            +- PhotonShuffleExchangeSink (3)
               +- PhotonProject (2)
                  +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [id#12073L AS order_id#12074L, (((id#12073L * 17) % 10000) + 1) AS customer_id#12076L, date_add(2025-01-01, cast((id#12073L % 365) as int)) AS order_date#12078, element_at([COMPLETED,COMPLETED,COMPLETED,CANCELLED], cast(((id#12073L % 4) + 1) as int), None, true) AS order_status#12080]

(3) PhotonShuffleExchangeSink
Input [4]: [order_id#12074L, customer_id#12076L, order_date#12078, order_status#12080]
Arguments: hashpartitioning(customer_id#12076L, 8)

(4) PhotonShuffleMapStage
Input [4]: [order_id#12

partition_id,count
0,61650
1,62450
2,61350
3,64650
4,62550
5,62350
6,61550
7,63450


Partitions : 8
Min rows   : 61350
Max rows   : 64650
Avg rows   : 62500.0


DataFrame[partition_id: int, count: bigint]

# 6. `coalesce()`

`coalesce()` is primarily used to reduce partitions and generally avoids the full
redistribution associated with `repartition()`.

We compare the physical plans rather than relying on the RDD API.

In [0]:
orders_16 = orders.repartition(16)

orders_4_coalesced = orders_16.coalesce(4)
orders_4_repartitioned = orders_16.repartition(4)

print("=== COALESCE(4) PLAN ===")
orders_4_coalesced.explain("formatted")

print("=== REPARTITION(4) PLAN ===")
orders_4_repartitioned.explain("formatted")

=== COALESCE(4) PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   Coalesce (9)
   +- ColumnarToRow (8)
      +- PhotonResultStage (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonSort (3)
                     +- PhotonProject (2)
                        +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [id#12073L AS order_id#12074L, (((id#12073L * 17) % 10000) + 1) AS customer_id#12076L, date_add(2025-01-01, cast((id#12073L % 365) as int)) AS order_date#12078, element_at([COMPLETED,COMPLETED,COMPLETED,CANCELLED], cast(((id#12073L % 4) + 1) as int), None, true) AS order_status#12080]

(3) PhotonSort
Input [4]: [order_id#12074L, customer_id#12076L, order_date#12078, order_status#12080]
Arguments: [order_id#12074L ASC NULLS FIRST, customer_

### What to look for

`repartition()` should normally show an `Exchange` because data must be redistributed.

`coalesce()` is designed to reduce partitions with less redistribution.

The exact physical plan can vary with the Databricks runtime and optimizer.

# 7. Shuffle — what makes Spark move data?

A shuffle occurs when Spark needs to redistribute data across partitions.

Typical examples:
- `groupBy`
- `distinct`
- `dropDuplicates`
- `orderBy`
- `repartition`
- many joins

In physical plans, look for `Exchange`.

Shuffle is not automatically bad. The engineering question is whether it is necessary,
appropriately sized, and reasonably balanced.

In [0]:
customer_order_counts = (
    orders
    .groupBy("customer_id")
    .count()
)

print("=== GROUP BY PLAN ===")
customer_order_counts.explain("formatted")

assert customer_order_counts.count() == NUM_CUSTOMERS
print("GROUP BY validation: PASS")

=== GROUP BY PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [(((id#12073L * 17) % 10000) + 1) AS customer_id#12076L]

(3) PhotonGroupingAgg
Input [1]: [customer_id#12076L]
Arguments: [customer_id#12076L], [partial_count(1) AS count#12446L], [count#12445L], [customer_id#12076L, count#12446L], false

(4) PhotonShuffleExchangeSink
Input [2]: [customer_id#12076L, count#12446L]
Arguments: hashpartitioning(customer_id#12076L, 16)

(5) PhotonShuffleMapStage
Input [2]: [customer_id#12076

In [0]:
distinct_customers = (
    orders
    .select("customer_id")
    .distinct()
)

print("=== DISTINCT PLAN ===")
distinct_customers.explain("formatted")

assert distinct_customers.count() == NUM_CUSTOMERS

=== DISTINCT PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [(((id#12073L * 17) % 10000) + 1) AS customer_id#12076L]

(3) PhotonGroupingAgg
Input [1]: [customer_id#12076L]
Arguments: [customer_id#12076L], [customer_id#12076L], false

(4) PhotonShuffleExchangeSink
Input [1]: [customer_id#12076L]
Arguments: hashpartitioning(customer_id#12076L, 16)

(5) PhotonShuffleMapStage
Input [1]: [customer_id#12076L]
Arguments: ENSURE_REQUIREMENTS, [id=#27316]

(6) PhotonShuffleExchangeSource


In [0]:
unique_orders = orders.dropDuplicates(["order_id"])

print("=== DROP DUPLICATES PLAN ===")
unique_orders.explain("formatted")

assert unique_orders.count() == NUM_ORDERS
print("Deduplication validation: PASS")

=== DROP DUPLICATES PLAN ===
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [id#12073L AS order_id#12074L, (((id#12073L * 17) % 10000) + 1) AS customer_id#12464L, date_add(2025-01-01, cast((id#12073L % 365) as int)) AS order_date#12466, element_at([COMPLETED,COMPLETED,COMPLETED,CANCELLED], cast(((id#12073L % 4) + 1) as int), None, true) AS order_status#12468]

(3) PhotonColumnarToRow
Input [4]: [order_id#12074L, customer_id#12464L, order_date#12466, order_status#12468]

(4) PhotonResultStage
Input [4]: [order_id#12074L, customer_id#12464L, order_date#12466, order_status#12468]


== Photon Explanation ==
The query is fully supported by Photon.
Deduplication validation: PASS


In [0]:
sorted_orders = orders.orderBy("customer_id")

print("=== ORDER BY PLAN ===")
sorted_orders.explain("formatted")

=== ORDER BY PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- == Initial Plan ==
   PhotonResultStage (8)
   +- PhotonColumnarToRow (7)
      +- PhotonSort (6)
         +- PhotonShuffleExchangeSource (5)
            +- PhotonShuffleMapStage (4)
               +- PhotonShuffleExchangeSink (3)
                  +- PhotonProject (2)
                     +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [id#12073L AS order_id#12074L, (((id#12073L * 17) % 10000) + 1) AS customer_id#12076L, date_add(2025-01-01, cast((id#12073L % 365) as int)) AS order_date#12078, element_at([COMPLETED,COMPLETED,COMPLETED,CANCELLED], cast(((id#12073L % 4) + 1) as int), None, true) AS order_status#12080]

(3) PhotonShuffleExchangeSink
Input [4]: [order_id#12074L, customer_id#12076L, order_date#12078, order_status#12080]
Arguments: rangepartitioning(customer_id#12076L ASC NULLS FIRST, 16)

(4) Pho

# 8. Physical plan reading

Focus on these operators:

- `Exchange` — redistribution/shuffle
- `SortMergeJoin` — shuffle/sort-based join strategy
- `BroadcastHashJoin` — broadcast join
- `HashAggregate` — aggregation
- `AdaptiveSparkPlan` — AQE involvement
- `AQEShuffleRead` — adaptive shuffle reading

The exact plan depends on Spark/Databricks runtime, statistics, and the environment.

In [0]:
print("=== BASELINE PLAN ===")
baseline_customer_revenue.explain("formatted")

=== BASELINE PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (44)
+- == Initial Plan ==
   PhotonResultStage (43)
   +- PhotonColumnarToRow (42)
      +- PhotonSort (41)
         +- PhotonShuffleExchangeSource (40)
            +- PhotonShuffleMapStage (39)
               +- PhotonShuffleExchangeSink (38)
                  +- PhotonGroupingAgg (37)
                     +- PhotonShuffleExchangeSource (36)
                        +- PhotonShuffleMapStage (35)
                           +- PhotonShuffleExchangeSink (34)
                              +- PhotonGroupingAgg (33)
                                 +- PhotonGroupingAgg (32)
                                    +- PhotonGroupingAgg (31)
                                       +- PhotonProject (30)
                                          +- PhotonBroadcastHashJoin Inner (29)
                                             :- PhotonProject (23)
                                             :  +- PhotonBroadcastHashJoin Inner (22)
        

# 9. Broadcast join

A small dimension table can often be broadcast to executors.

This can avoid shuffling the large side of the join.

We compare:
1. normal join
2. explicit broadcast join

Do not assume broadcast is always better; the small side must actually be suitable.

In [0]:
normal_customer_join = (
    orders
    .join(
        customers.select("customer_id", "city"),
        on="customer_id",
        how="inner"
    )
)

broadcast_customer_join = (
    orders
    .join(
        broadcast(customers.select("customer_id", "city")),
        on="customer_id",
        how="inner"
    )
)

print("=== NORMAL CUSTOMER JOIN ===")
normal_customer_join.explain("formatted")

print("=== BROADCAST CUSTOMER JOIN ===")
broadcast_customer_join.explain("formatted")

assert normal_customer_join.count() == NUM_ORDERS
assert broadcast_customer_join.count() == NUM_ORDERS

print("Broadcast join correctness: PASS")

=== NORMAL CUSTOMER JOIN ===
== Physical Plan ==
AdaptiveSparkPlan (12)
+- == Initial Plan ==
   PhotonResultStage (11)
   +- PhotonColumnarToRow (10)
      +- PhotonProject (9)
         +- PhotonBroadcastHashJoin Inner (8)
            :- PhotonProject (2)
            :  +- PhotonRange (1)
            +- PhotonShuffleExchangeSource (7)
               +- PhotonShuffleMapStage (6)
                  +- PhotonShuffleExchangeSink (5)
                     +- PhotonProject (4)
                        +- PhotonRange (3)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [id#12073L AS order_id#12074L, (((id#12073L * 17) % 10000) + 1) AS customer_id#12076L, date_add(2025-01-01, cast((id#12073L % 365) as int)) AS order_date#12078, element_at([COMPLETED,COMPLETED,COMPLETED,CANCELLED], cast(((id#12073L % 4) + 1) as int), None, true) AS order_status#12080]

(3) PhotonRange
Output [1]: [id#12501L]
Arguments: Ra

In [0]:
normal_join_metrics = benchmark_query(
    normal_customer_join,
    "Normal Customer Join"
)

broadcast_join_metrics = benchmark_query(
    broadcast_customer_join,
    "Broadcast Customer Join"
)

Normal Customer Join | Run 1 | Rows: 500,000 | Time: 0.32s
Normal Customer Join | Run 2 | Rows: 500,000 | Time: 0.29s
Broadcast Customer Join | Run 1 | Rows: 500,000 | Time: 0.38s
Broadcast Customer Join | Run 2 | Rows: 500,000 | Time: 0.33s


# 10. Large-table join / shuffle strategy

For larger inputs, Spark may use a shuffle-based join such as `SortMergeJoin`.

We deliberately create two larger DataFrames and inspect the resulting physical plan.

In [0]:
orders_for_join = (
    orders
    .select(
        "order_id",
        "customer_id",
        "order_status"
    )
)

orders_copy = (
    orders
    .select(
        F.col("order_id").alias("order_id_2"),
        F.col("customer_id").alias("customer_id_2")
    )
)

large_join = (
    orders_for_join
    .join(
        orders_copy,
        orders_for_join.customer_id == orders_copy.customer_id_2,
        how="inner"
    )
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_id_2"
    )
)

print("=== LARGE-LARGE JOIN PLAN ===")
large_join.explain("formatted")

=== LARGE-LARGE JOIN PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (15)
+- == Initial Plan ==
   PhotonResultStage (14)
   +- PhotonColumnarToRow (13)
      +- PhotonProject (12)
         +- PhotonShuffledHashJoin Inner (11)
            :- PhotonShuffleExchangeSource (5)
            :  +- PhotonShuffleMapStage (4)
            :     +- PhotonShuffleExchangeSink (3)
            :        +- PhotonProject (2)
            :           +- PhotonRange (1)
            +- PhotonShuffleExchangeSource (10)
               +- PhotonShuffleMapStage (9)
                  +- PhotonShuffleExchangeSink (8)
                     +- PhotonProject (7)
                        +- PhotonRange (6)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [id#12073L AS order_id#12074L, (((id#12073L * 17) % 10000) + 1) AS customer_id#12076L, element_at([COMPLETED,COMPLETED,COMPLETED,CANCELLED], cast(((id#12073L % 4) + 1) as int), 

# 11. Data skew

Data skew occurs when a small number of keys receive a disproportionately large
amount of data.

Simply increasing partition count does not necessarily solve hot-key skew because
the same hot key can continue to hash to one partition.

We create a controlled skewed dataset with one hot customer.

In [0]:
HOT_CUSTOMER_ID = 1
SKEWED_ORDERS = 500_000
HOT_ROWS = 250_000

skewed_orders = (
    spark.range(1, SKEWED_ORDERS + 1)
    .withColumnRenamed("id", "order_id")
    .withColumn(
        "customer_id",
        F.when(
            F.col("order_id") <= HOT_ROWS,
            F.lit(HOT_CUSTOMER_ID)
        ).otherwise(
            # Generate only customer IDs 2..NUM_CUSTOMERS
            ((F.col("order_id") * 17) % (NUM_CUSTOMERS - 1)) + 2
        )
    )
)

print("=== SKEW DISTRIBUTION ===")

skew_distribution = (
    skewed_orders
    .groupBy("customer_id")
    .count()
    .orderBy(F.col("count").desc())
)

display(skew_distribution.limit(10))

hot_count = (
    skew_distribution
    .filter(F.col("customer_id") == HOT_CUSTOMER_ID)
    .select("count")
    .first()["count"]
)

print("Hot customer rows:", hot_count)

assert hot_count == HOT_ROWS

print("Skew generation validation: PASS")

=== SKEW DISTRIBUTION ===


customer_id,count
1,250000
461,26
716,26
512,26
631,26
597,26
733,26
444,26
478,26
801,26


Hot customer rows: 250000
Skew generation validation: PASS


In [0]:
assert skewed_orders.count() == SKEWED_ORDERS

print("Total skewed orders:", skewed_orders.count())
print("Hot customer rows:", hot_count)
print("Skew generation: PASS")

Total skewed orders: 500000
Hot customer rows: 250000
Skew generation: PASS


In [0]:
skewed_partitioned = skewed_orders.repartition(8, "customer_id")

print("=== SKEWED REPARTITION PLAN ===")
skewed_partitioned.explain("formatted")

skew_partition_distribution = partition_distribution(
    skewed_partitioned,
    "Skewed Orders"
)

=== SKEWED REPARTITION PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (8)
+- == Initial Plan ==
   PhotonResultStage (7)
   +- PhotonColumnarToRow (6)
      +- PhotonShuffleExchangeSource (5)
         +- PhotonShuffleMapStage (4)
            +- PhotonShuffleExchangeSink (3)
               +- PhotonProject (2)
                  +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#12585L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12585L]
Arguments: [id#12585L AS order_id#12586L, CASE WHEN (id#12585L <= 250000) THEN 1 ELSE (((id#12585L * 17) % 9999) + 2) END AS customer_id#12588L]

(3) PhotonShuffleExchangeSink
Input [2]: [order_id#12586L, customer_id#12588L]
Arguments: hashpartitioning(customer_id#12588L, 8)

(4) PhotonShuffleMapStage
Input [2]: [order_id#12586L, customer_id#12588L]
Arguments: REPARTITION_BY_NUM, [id=#30154]

(5) PhotonShuffleExchangeSource
Input [2]: [order_id#12586L, customer_id#12588L]

(6) PhotonColumnarToRow
Input [2]: [order_i

partition_id,count
0,30828
1,31229
2,30677
3,32328
4,31280
5,281151
6,30778
7,31729


Partitions : 8
Min rows   : 30677
Max rows   : 281151
Avg rows   : 62500.0


# 12. Salting — skew mitigation

Salting adds a secondary key to distribute records belonging to a hot key across
multiple partitions.

This is a data/query design technique and should be used carefully because the join
logic must preserve correctness.

In [0]:
SALT_BUCKETS = 8

skewed_orders_salted = (
    skewed_orders
    .withColumn(
        "salt",
        F.when(
            F.col("customer_id") == HOT_CUSTOMER_ID,
            F.pmod(F.col("order_id"), F.lit(SALT_BUCKETS)).cast("int")
        ).otherwise(
            F.lit(0)
        )
    )
)

print("=== SALTED DATA DISTRIBUTION ===")

salt_distribution = (
    skewed_orders_salted
    .groupBy("customer_id", "salt")
    .count()
    .orderBy(
        F.col("customer_id"),
        F.col("salt")
    )
)

display(
    salt_distribution
    .filter(F.col("customer_id") == HOT_CUSTOMER_ID)
)

=== SALTED DATA DISTRIBUTION ===


customer_id,salt,count
1,0,31250
1,1,31250
1,2,31250
1,3,31250
1,4,31250
1,5,31250
1,6,31250
1,7,31250


# 13. Adaptive Query Execution (AQE)

AQE allows Spark to use runtime information to adapt query execution.

Important capabilities include:
- post-shuffle partition coalescing
- skew join handling
- runtime plan adaptation

### Serverless approach

This notebook does **not** query or set:

```python
spark.sql.adaptive.enabled
```

because that configuration is not available through the current Serverless configuration
interface.

Instead, AQE is evaluated through the actual physical execution plan.
Look for:
- `AdaptiveSparkPlan`
- `AQEShuffleRead`

In [0]:
aqe_customer_counts = (
    orders
    .groupBy("customer_id")
    .count()
)

print("=== AQE / AGGREGATION PLAN ===")
aqe_customer_counts.explain("formatted")

aqe_result = aqe_customer_counts.count()

assert aqe_result == NUM_CUSTOMERS
print("AQE experiment validation: PASS")

=== AQE / AGGREGATION PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (10)
+- == Initial Plan ==
   PhotonResultStage (9)
   +- PhotonColumnarToRow (8)
      +- PhotonGroupingAgg (7)
         +- PhotonShuffleExchangeSource (6)
            +- PhotonShuffleMapStage (5)
               +- PhotonShuffleExchangeSink (4)
                  +- PhotonGroupingAgg (3)
                     +- PhotonProject (2)
                        +- PhotonRange (1)


(1) PhotonRange
Output [1]: [id#12073L]
Arguments: Range (1, 500001, step=1, splits=8)

(2) PhotonProject
Input [1]: [id#12073L]
Arguments: [(((id#12073L * 17) % 10000) + 1) AS customer_id#12076L]

(3) PhotonGroupingAgg
Input [1]: [customer_id#12076L]
Arguments: [customer_id#12076L], [partial_count(1) AS count#12670L], [count#12669L], [customer_id#12076L, count#12670L], false

(4) PhotonShuffleExchangeSink
Input [2]: [customer_id#12076L, count#12670L]
Arguments: hashpartitioning(customer_id#12076L, 16)

(5) PhotonShuffleMapStage
Input [2]: [customer

# 14. Performance comparison

In [0]:
performance_results = [
    baseline_metrics,
    normal_join_metrics,
    broadcast_join_metrics
]

performance_df = spark.createDataFrame(performance_results)

display(
    performance_df.orderBy("avg_seconds")
)

avg_seconds,experiment,min_seconds,rows
0.3071688460000246,Normal Customer Join,0.29258235500003593,500000
0.3533343595000247,Broadcast Customer Join,0.33028789000002234,500000
0.7013309475000256,Baseline Customer Revenue,0.6472234200000457,10000


### Interpreting the benchmark

Use runtime as an **observed measurement**, not a universal benchmark.

Serverless resource availability, optimizer behavior, data statistics, and workload
state can affect individual runs.

The more important evidence is the combination of:

1. physical plan
2. shuffle/exchange behavior
3. partition distribution
4. join strategy
5. runtime
6. correctness

# 15. Correctness reconciliation

In [0]:
baseline_total = (
    baseline_customer_revenue
    .agg(F.sum("total_revenue").alias("total"))
    .first()["total"]
)

item_total = (
    item_revenue
    .agg(F.sum("revenue").alias("total"))
    .first()["total"]
)

print("Baseline aggregated revenue:", baseline_total)
print("Item-level revenue:", item_total)

assert baseline_total == item_total
print("Revenue reconciliation: PASS")

Baseline aggregated revenue: 9742500000.0
Item-level revenue: 9742500000.0
Revenue reconciliation: PASS


# 16. Serverless compatibility quality gate

In [0]:
print("=" * 70)
print("SERVERLESS COMPATIBILITY CHECK")
print("=" * 70)

print("✓ DataFrame API used")
print("✓ No Python RDD API")
print("✓ No spark.sql.adaptive.enabled dependency")
print("✓ element_at indexes explicitly cast to INT")
print("✓ Physical plans inspected with explain('formatted')")
print("✓ Partition distribution inspected with spark_partition_id()")
print("✓ Broadcast join demonstrated")
print("✓ Shuffle demonstrated")
print("✓ Data skew demonstrated")
print("✓ Salting demonstrated")
print("✓ AQE inspected through physical plan")
print("✓ Runtime benchmarking included")
print("✓ Correctness validation included")

print("=" * 70)
print("FINAL QUALITY GATE: PASS")
print("=" * 70)

SERVERLESS COMPATIBILITY CHECK
✓ DataFrame API used
✓ No Python RDD API
✓ No spark.sql.adaptive.enabled dependency
✓ element_at indexes explicitly cast to INT
✓ Physical plans inspected with explain('formatted')
✓ Partition distribution inspected with spark_partition_id()
✓ Broadcast join demonstrated
✓ Shuffle demonstrated
✓ Data skew demonstrated
✓ Salting demonstrated
✓ AQE inspected through physical plan
✓ Runtime benchmarking included
✓ Correctness validation included
FINAL QUALITY GATE: PASS


# 17. Practical Spark optimization guidelines

### Partitioning
- More partitions are not automatically better.
- Too few partitions can reduce parallelism.
- Too many can create scheduling/small-task overhead.
- Inspect distribution rather than guessing.

### `repartition()`
- Changes partition count/distribution.
- Usually introduces shuffle.
- Useful when downstream work benefits from a particular distribution.

### `coalesce()`
- Primarily reduces partitions.
- Usually avoids a full redistribution.
- Useful after filtering or when reducing output partitions.

### Shuffle
- Look for `Exchange`.
- Shuffle is often necessary.
- Optimize unnecessary, excessive, or highly skewed shuffle.

### Broadcast
- Useful when one side is genuinely small.
- Inspect the physical plan for `BroadcastHashJoin`.
- Do not blindly broadcast large tables.

### Data skew
- One hot key can dominate a partition.
- Increasing partition count alone may not solve hot-key skew.
- Consider AQE skew handling, salting, or data-model redesign.

### AQE
- Use runtime plan evidence.
- On Serverless, do not depend on querying unsupported configuration keys.
- Inspect `AdaptiveSparkPlan` and `AQEShuffleRead`.

### Engineering workflow

**Inspect → Diagnose → Optimize → Measure → Validate**

# 18. Final project summary

This notebook demonstrates a practical Spark performance investigation using a
controlled e-commerce workload.

It covers:

- physical execution plans
- partition distribution
- `repartition()`
- `coalesce()`
- shuffle and `Exchange`
- broadcast joins
- large-table joins
- data skew
- salting
- Adaptive Query Execution
- runtime benchmarking
- correctness reconciliation

The implementation is intentionally compatible with **Databricks Serverless Compute**
and avoids unsupported Python RDD APIs.